# Credit Card Fraud Detection
### Fixed & Improved Version

## 1. Import Dependencies

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load the Dataset
> **Fix 1:** Upload the file via Colab's file uploader instead of hardcoding `/content/` path.

In [ ]:
# Option A: Upload manually via Colab
from google.colab import files
uploaded = files.upload()  # Select your credit_data.csv file

# Load into DataFrame
credit_card_data = pd.read_csv('credit_data.csv')
print("Dataset loaded successfully!")
print(f"Shape: {credit_card_data.shape}")

## 3. Explore the Dataset

In [ ]:
credit_card_data.head()

In [ ]:
credit_card_data.tail()

In [ ]:
credit_card_data.info()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(credit_card_data.isnull().sum())

In [ ]:
# Class distribution
print("Class distribution:")
print(credit_card_data['Class'].value_counts())
print()
print("0 --> Normal Transaction")
print("1 --> Fraudulent Transaction")

## 4. Analyse the Imbalance
> This dataset is highly imbalanced — far more normal transactions than fraudulent ones.

In [ ]:
legit = credit_card_data[credit_card_data.Class == 0]
fraud = credit_card_data[credit_card_data.Class == 1]
print(f"Legit transactions:  {legit.shape[0]}")
print(f"Fraud transactions:  {fraud.shape[0]}")

In [ ]:
print("Legit transaction amounts:")
print(legit.Amount.describe())
print()
print("Fraud transaction amounts:")
print(fraud.Amount.describe())

In [ ]:
credit_card_data.groupby('Class').mean()

## 5. Under-Sampling
> Build a balanced dataset with equal numbers of legit and fraudulent transactions.

In [ ]:
# Sample 492 legit transactions to match fraud count
legit_sample = legit.sample(n=492, random_state=2)

# Combine into new balanced dataset
new_dataset = pd.concat([legit_sample, fraud], axis=0)
print(f"New dataset shape: {new_dataset.shape}")
print()
print("New class distribution:")
print(new_dataset['Class'].value_counts())

## 6. Split Features and Target

In [ ]:
# FIX 2: Use either 'columns=' OR 'axis=', not both
X = new_dataset.drop(columns='Class')   # was: drop(columns='Class', axis=1) — INVALID
Y = new_dataset['Class']
print(f"Features shape: {X.shape}")
print(f"Target shape:   {Y.shape}")

## 7. Feature Scaling
> **Improvement:** StandardScaler improves Logistic Regression performance significantly.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Features scaled successfully.")

## 8. Train-Test Split

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X_scaled, Y, test_size=0.2, stratify=Y, random_state=2
)
print(f"Total samples:    {X_scaled.shape[0]}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

## 9. Model Training — Logistic Regression

In [ ]:
# FIX 3: Set max_iter=1000 to ensure convergence
model = LogisticRegression(max_iter=1000)
model.fit(X_train, Y_train)
print("Model trained successfully.")

## 10. Model Evaluation

In [ ]:
# FIX 4: Correct argument order -> accuracy_score(y_true, y_pred)
X_train_prediction = model.predict(X_train)
training_accuracy = accuracy_score(Y_train, X_train_prediction)  # was swapped
print(f"Accuracy on Training Data: {training_accuracy:.4f}")

In [ ]:
X_test_prediction = model.predict(X_test)
test_accuracy = accuracy_score(Y_test, X_test_prediction)  # was swapped
print(f"Accuracy on Test Data:     {test_accuracy:.4f}")

## 11. Detailed Evaluation Metrics
> **Improvement:** For fraud detection, precision, recall and F1-score matter more than accuracy alone.

In [ ]:
print("=== Classification Report (Test Data) ===")
print(classification_report(Y_test, X_test_prediction, target_names=['Legit', 'Fraud']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(Y_test, X_test_prediction)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Legit', 'Predicted Fraud'],
            yticklabels=['Actual Legit', 'Actual Fraud'])
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

print(f"True Negatives  (Legit correctly identified):  {cm[0][0]}")
print(f"False Positives (Legit flagged as Fraud):       {cm[0][1]}")
print(f"False Negatives (Fraud missed):                 {cm[1][0]}")
print(f"True Positives  (Fraud correctly caught):       {cm[1][1]}")